# 05 — Gold Aggregations — Corrected Week 07

**Week:** 7  
**Date:** September 1–7, 2026  
**Goal:** Build governed Gold facts and KPI summaries from trusted data, prove grain and reconciliation, and leave a Gold-only contract ready for review.

> **Important:** This notebook follows the approved P10 ShipTrack Gold model and KPI contracts. All Gold tables use the `gold_*` prefix per the approved naming convention.

**Execution rule:** Run from top to bottom in Databricks. `CREATE OR REPLACE TABLE ... USING DELTA` is used for deterministic reruns.

## Week 07 scope and approved Gold objects

### Gold facts
- `gold_fact_shipment` — one trusted shipment
- `gold_fact_shipment_scan` — one trusted historical scan
- `gold_fact_shipment_exception` — one trusted exception occurrence
- `gold_fact_scan_event` — reserved for Week 10 controlled streaming; not built here

### Gold summaries
- `gold_shipment_delay_summary`
- `gold_hub_throughput_summary`
- `gold_carrier_performance_summary`
- `gold_route_reliability_summary`
- `gold_shipment_status_exception_summary`

### KPI contracts covered
- Total Shipments
- Delivered Shipments
- On-Time Delivery Rate
- Average Delivery Delay — **late delivered only**
- Average End-to-End Transit Time
- Exception Rate
- First-Attempt Delivery Success Rate
- Hub Average Dwell Time

**Grain rule:** shipment KPIs use distinct `shipment_id`; scan KPIs use scan/event grain. Shipment measures are never summed after a one-to-many scan or exception join.

## 1. Source handoff and compatibility discovery

The existing team notebook uses `trusted_shipments`. The approved playbook names the canonical source `trusted_silver_shipments`. The discovery cell below accepts either naming convention and records which objects are actually present in the Databricks workspace.

For scans and exceptions, the cell checks the approved Trusted Silver names first and then compatible team naming variants. If no trusted source exists, execution stops with an explicit message instead of silently using Silver Candidate or raw data.

In [0]:
from pyspark.sql import functions as F

# Approved source first, then the team naming already used by the existing Week-7 notebook.
source_candidates = {
    "shipments": ["trusted_silver_shipments", "trusted_shipments"],
    "scans": ["trusted_silver_scans", "trusted_scans", "trusted_scan_events"],
    "exceptions": ["trusted_silver_exceptions", "trusted_exceptions"]
}

def first_existing(candidates):
    for name in candidates:
        if spark.catalog.tableExists(name):
            return name
    return None

resolved = {k: first_existing(v) for k, v in source_candidates.items()}
print("Resolved Trusted sources:")
for k, v in resolved.items():
    print(f"  {k}: {v}")

if not resolved["shipments"]:
    raise RuntimeError("No trusted shipment source found. Expected trusted_silver_shipments or trusted_shipments.")
if not resolved["scans"]:
    raise RuntimeError("No trusted historical scan source found. Expected trusted_silver_scans/trusted_scans/trusted_scan_events.")
if not resolved["exceptions"]:
    raise RuntimeError("No trusted exception source found. Expected trusted_silver_exceptions or trusted_exceptions.")

trusted_shipment_source = resolved["shipments"]
trusted_scan_source = resolved["scans"]
trusted_exception_source = resolved["exceptions"]

# Expose stable temporary names for the SQL cells below.
spark.table(trusted_shipment_source).createOrReplaceTempView("w07_trusted_shipments")
spark.table(trusted_scan_source).createOrReplaceTempView("w07_trusted_scans_raw")
spark.table(trusted_exception_source).createOrReplaceTempView("w07_trusted_exceptions_raw")

print("Week-7 trusted-source handoff ready.")

Resolved Trusted sources:
  shipments: trusted_shipments
  scans: trusted_scan_events
  exceptions: trusted_exceptions
Week-7 trusted-source handoff ready.


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_carriers,false
default,bronze_exceptions,false
default,bronze_hubs,false
default,bronze_routes,false
default,bronze_scan_events,false
default,bronze_shipments,false
default,candidate_carriers,false
default,candidate_exceptions,false
default,candidate_hubs,false
default,candidate_routes,false


## 2. Normalize the trusted scan and exception inputs

The approved model requires a stable Gold grain even if the team's Trusted Silver column names differ slightly. The Python cell below inspects the resolved schemas and creates normalized temporary views. It does **not** fall back to raw, Bronze, Silver Candidate, or Quarantine.

In [0]:
from pyspark.sql import functions as F

def pick_col(columns, candidates, label, required=True):
    cols = set(columns)
    for c in candidates:
        if c in cols:
            return c
    if required:
        raise RuntimeError(f"Could not resolve {label}. Available columns: {sorted(cols)}")
    return None

scan_df = spark.table(trusted_scan_source)
exc_df = spark.table(trusted_exception_source)

# Compute column lists once to avoid repeated Analyze RPCs (Spark Connect)
scan_cols = scan_df.columns
exc_cols = exc_df.columns

scan_id_col = pick_col(scan_cols, ["scan_id", "event_id"], "scan identifier")
scan_shipment_col = pick_col(scan_cols, ["shipment_id", "reference_id"], "scan shipment_id")
scan_seq_col = pick_col(scan_cols, ["event_sequence_no", "sequence_no", "scan_sequence_no"], "scan sequence")
scan_status_col = pick_col(scan_cols, ["shipment_status", "scan_status", "status", "status_standard"], "scan status")
scan_time_col = pick_col(scan_cols, ["event_time", "event_timestamp", "scan_timestamp"], "scan event timestamp")
scan_hub_col = pick_col(scan_cols, ["hub_id", "event_hub_id"], "scan hub_id")
scan_carrier_col = pick_col(scan_cols, ["carrier_id"], "scan carrier_id", required=False)
scan_route_col = pick_col(scan_cols, ["route_id"], "scan route_id", required=False)
scan_event_type_col = pick_col(scan_cols, ["event_type"], "scan event_type", required=False)
scan_package_col = pick_col(scan_cols, ["package_condition"], "scan package_condition", required=False)

scan_selects = [
    F.col(scan_id_col).alias("scan_id"),
    F.col(scan_shipment_col).alias("shipment_id"),
    F.col(scan_seq_col).cast("int").alias("event_sequence_no"),
    F.col(scan_status_col).alias("shipment_status"),
    F.col(scan_time_col).cast("timestamp").alias("event_time"),
    F.col(scan_hub_col).alias("hub_id"),
]
if scan_carrier_col:
    scan_selects.append(F.col(scan_carrier_col).alias("carrier_id"))
if scan_route_col:
    scan_selects.append(F.col(scan_route_col).alias("route_id"))
if scan_event_type_col:
    scan_selects.append(F.col(scan_event_type_col).alias("event_type"))
if scan_package_col:
    scan_selects.append(F.col(scan_package_col).alias("package_condition"))

scan_norm = scan_df.select(*scan_selects)
scan_norm.createOrReplaceTempView("w07_trusted_scans")

exc_id_col = pick_col(exc_cols, ["exception_id", "exception_occurrence_id", "source_record_id"], "exception identifier")
exc_shipment_col = pick_col(exc_cols, ["shipment_id"], "exception shipment_id")
exc_type_col = pick_col(exc_cols, ["exception_type", "exception_category", "type"], "exception type")
exc_time_col = pick_col(exc_cols, ["exception_ts", "exception_timestamp", "exception_time", "event_time", "created_ts", "created_at"], "exception timestamp")
exc_hub_col = pick_col(exc_cols, ["hub_id"], "exception hub_id", required=False)
exc_severity_col = pick_col(exc_cols, ["severity"], "exception severity", required=False)
exc_resolution_col = pick_col(exc_cols, ["resolution_status"], "exception resolution_status", required=False)
exc_resolution_time_col = pick_col(exc_cols, ["resolution_time"], "exception resolution_time", required=False)
exc_scan_col = pick_col(exc_cols, ["scan_id"], "exception scan_id", required=False)

exc_selects = [
    F.col(exc_id_col).alias("exception_id"),
    F.col(exc_shipment_col).alias("shipment_id"),
    F.col(exc_type_col).alias("exception_type"),
    F.col(exc_time_col).cast("timestamp").alias("exception_ts"),
]
if exc_hub_col:
    exc_selects.append(F.col(exc_hub_col).alias("hub_id"))
if exc_severity_col:
    exc_selects.append(F.col(exc_severity_col).alias("severity"))
if exc_resolution_col:
    exc_selects.append(F.col(exc_resolution_col).alias("resolution_status"))
if exc_resolution_time_col:
    exc_selects.append(F.col(exc_resolution_time_col).cast("timestamp").alias("resolution_time"))
if exc_scan_col:
    exc_selects.append(F.col(exc_scan_col).alias("scan_id"))

exc_norm = exc_df.select(*exc_selects)
exc_norm.createOrReplaceTempView("w07_trusted_exceptions")

print("Normalized trusted scan columns:", scan_norm.columns)
print("Normalized trusted exception columns:", exc_norm.columns)

Normalized trusted scan columns: ['scan_id', 'shipment_id', 'event_sequence_no', 'shipment_status', 'event_time', 'hub_id', 'carrier_id', 'route_id', 'event_type', 'package_condition']
Normalized trusted exception columns: ['exception_id', 'shipment_id', 'exception_type', 'exception_ts', 'hub_id', 'severity', 'resolution_status', 'resolution_time', 'scan_id']


## 3. GOLD-01 — `gold_fact_shipment`

**Grain:** exactly one row per trusted `shipment_id`.

The fact carries shipment measures and derives reusable KPI fields once, at shipment grain. This prevents shipment measures from multiplying when scans or exceptions are later joined.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_shipment USING DELTA AS
WITH shipment_base AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY source_record_id) AS rn
    FROM w07_trusted_shipments
)
SELECT
    source_record_id,
    shipment_id,
    order_reference,
    route_id,
    origin_hub_id,
    destination_hub_id,
    carrier_id,
    service_level,
    transport_mode,
    booking_ts,
    pickup_ts,
    promised_delivery_ts,
    actual_delivery_ts,
    return_completed_ts,
    latest_status,
    delivery_outcome,
    status_reason,
    package_weight_kg,
    package_count,
    freight_amount_inr,
    attempt_count,
    exception_flag,
    CASE
        WHEN UPPER(delivery_outcome) = 'DELIVERED'
         AND actual_delivery_ts IS NOT NULL
         AND promised_delivery_ts IS NOT NULL
        THEN GREATEST(
            CAST((unix_timestamp(actual_delivery_ts) - unix_timestamp(promised_delivery_ts)) / 60 AS BIGINT),
            CAST(0 AS BIGINT)
        )
    END AS delivery_delay_minutes,
    CASE
        WHEN UPPER(delivery_outcome) = 'DELIVERED'
         AND actual_delivery_ts IS NOT NULL
         AND pickup_ts IS NOT NULL
        THEN (unix_timestamp(actual_delivery_ts) - unix_timestamp(pickup_ts)) / 3600.0
    END AS transit_hours,
    CASE
        WHEN UPPER(delivery_outcome) = 'DELIVERED'
         AND actual_delivery_ts IS NOT NULL
         AND promised_delivery_ts IS NOT NULL
         AND actual_delivery_ts <= promised_delivery_ts
        THEN 1 ELSE 0 END AS on_time_flag,
    CASE
        WHEN UPPER(delivery_outcome) = 'DELIVERED' AND attempt_count = 1
        THEN 1 ELSE 0 END AS first_attempt_success_flag,
    CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END AS delivered_flag,
    CAST(booking_ts AS DATE) AS booking_date,
    CAST(actual_delivery_ts AS DATE) AS delivery_date
FROM shipment_base
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT shipment_id) AS distinct_shipments,
    COUNT(*) - COUNT(DISTINCT shipment_id) AS duplicate_shipment_rows
FROM gold_fact_shipment;

fact_rows,distinct_shipments,duplicate_shipment_rows
99857,99857,0


## 4. GOLD-02 — `gold_fact_shipment_scan`

**Grain:** one trusted historical scan.

The fact preserves `scan_id`, shipment sequence, event time, status and event hub so hub throughput and dwell can be calculated without changing shipment grain.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_shipment_scan USING DELTA AS
SELECT
    scan_id,
    shipment_id,
    event_sequence_no,
    shipment_status,
    event_time,
    hub_id,
    carrier_id,
    route_id,
    event_type,
    package_condition,
    CAST(event_time AS DATE) AS event_date
FROM w07_trusted_scans;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS fact_scan_rows,
    COUNT(DISTINCT scan_id) AS distinct_scan_ids,
    COUNT(*) - COUNT(DISTINCT scan_id) AS duplicate_scan_rows,
    SUM(CASE WHEN shipment_id IS NULL THEN 1 ELSE 0 END) AS null_shipment_ids
FROM gold_fact_shipment_scan;

fact_scan_rows,distinct_scan_ids,duplicate_scan_rows,null_shipment_ids
710666,710636,30,10


## 5. GOLD-03 — `gold_fact_shipment_exception`

**Grain:** one trusted exception occurrence.

Exception occurrences remain at exception grain. Exception rate returns to distinct `shipment_id` when it is used as a shipment KPI.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_shipment_exception USING DELTA AS
SELECT
    exception_id,
    shipment_id,
    exception_type,
    exception_ts,
    hub_id,
    severity,
    resolution_status,
    resolution_time,
    scan_id,
    CAST(exception_ts AS DATE) AS exception_date
FROM w07_trusted_exceptions;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS fact_exception_rows,
    COUNT(DISTINCT exception_id) AS distinct_exception_ids,
    COUNT(DISTINCT shipment_id) AS affected_shipments,
    SUM(CASE WHEN shipment_id IS NULL THEN 1 ELSE 0 END) AS null_shipment_ids
FROM gold_fact_shipment_exception;

fact_exception_rows,distinct_exception_ids,affected_shipments,null_shipment_ids
18330,18330,14791,0


## 6. Approved summary tables

Each summary has an explicit grain. Summary measures are derived from the appropriate fact only, with shipment KPIs staying at shipment grain.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_shipment_delay_summary USING DELTA AS
SELECT
    delivery_date,
    route_id,
    carrier_id,
    service_level,
    CASE
        WHEN delivery_delay_minutes = 0 THEN 'ON_TIME'
        WHEN delivery_delay_minutes BETWEEN 1 AND 60 THEN 'LATE_1_60_MIN'
        WHEN delivery_delay_minutes BETWEEN 61 AND 240 THEN 'LATE_61_240_MIN'
        ELSE 'LATE_240_PLUS_MIN'
    END AS delay_band,
    COUNT(DISTINCT shipment_id) AS shipment_count,
    ROUND(AVG(CASE WHEN delivery_delay_minutes > 0 THEN delivery_delay_minutes END), 2) AS avg_delay_minutes_late_only
FROM gold_fact_shipment
WHERE UPPER(delivery_outcome) = 'DELIVERED'
  AND delivery_date IS NOT NULL
GROUP BY delivery_date, route_id, carrier_id, service_level,
         CASE
            WHEN delivery_delay_minutes = 0 THEN 'ON_TIME'
            WHEN delivery_delay_minutes BETWEEN 1 AND 60 THEN 'LATE_1_60_MIN'
            WHEN delivery_delay_minutes BETWEEN 61 AND 240 THEN 'LATE_61_240_MIN'
            ELSE 'LATE_240_PLUS_MIN'
         END;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_carrier_performance_summary USING DELTA AS
SELECT
    delivery_date,
    carrier_id,
    service_level,
    COUNT(DISTINCT shipment_id) AS total_shipments,
    SUM(delivered_flag) AS delivered_shipments,
    SUM(CASE WHEN on_time_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END) AS on_time_shipments,
    ROUND(100.0 * SUM(CASE WHEN on_time_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS on_time_delivery_rate_pct,
    ROUND(AVG(transit_hours), 2) AS avg_transit_hours
FROM gold_fact_shipment
GROUP BY delivery_date, carrier_id, service_level;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_route_reliability_summary USING DELTA AS
SELECT
    delivery_date,
    route_id,
    service_level,
    COUNT(DISTINCT shipment_id) AS total_shipments,
    SUM(delivered_flag) AS delivered_shipments,
    SUM(CASE WHEN on_time_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END) AS on_time_shipments,
    ROUND(100.0 * SUM(CASE WHEN on_time_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS on_time_delivery_rate_pct,
    ROUND(AVG(delivery_delay_minutes), 2) AS avg_delivery_delay_minutes_all_delivered
FROM gold_fact_shipment
GROUP BY delivery_date, route_id, service_level;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_hub_throughput_summary USING DELTA AS
WITH sequenced AS (
    SELECT
        shipment_id, hub_id, event_time, shipment_status, event_date,
        LEAD(event_time) OVER (PARTITION BY shipment_id ORDER BY event_sequence_no, event_time, scan_id) AS next_event_time,
        LEAD(shipment_status) OVER (PARTITION BY shipment_id ORDER BY event_sequence_no, event_time, scan_id) AS next_status,
        LEAD(hub_id) OVER (PARTITION BY shipment_id ORDER BY event_sequence_no, event_time, scan_id) AS next_hub_id
    FROM gold_fact_shipment_scan
),
visits AS (
    SELECT
        event_date,
        hub_id,
        shipment_id,
        event_time AS arrival_ts,
        next_event_time AS departure_ts,
        (unix_timestamp(next_event_time) - unix_timestamp(event_time)) / 60.0 AS dwell_minutes
    FROM sequenced
    WHERE UPPER(shipment_status) IN ('AT_HUB','ARRIVED_AT_HUB','ARRIVAL')
      AND next_event_time IS NOT NULL
      AND next_hub_id = hub_id
      AND UPPER(next_status) IN ('OUT_FOR_DELIVERY','DEPARTED_HUB','DEPARTURE','IN_TRANSIT')
)
SELECT
    event_date,
    CASE
        WHEN HOUR(arrival_ts) < 6 THEN '00_06'
        WHEN HOUR(arrival_ts) < 12 THEN '06_12'
        WHEN HOUR(arrival_ts) < 18 THEN '12_18'
        ELSE '18_24'
    END AS time_band,
    hub_id,
    COUNT(DISTINCT shipment_id) AS shipment_count,
    ROUND(AVG(dwell_minutes), 2) AS avg_dwell_minutes
FROM visits
GROUP BY event_date,
         CASE
            WHEN HOUR(arrival_ts) < 6 THEN '00_06'
            WHEN HOUR(arrival_ts) < 12 THEN '06_12'
            WHEN HOUR(arrival_ts) < 18 THEN '12_18'
            ELSE '18_24'
         END, hub_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_shipment_status_exception_summary USING DELTA AS
WITH exception_rollup AS (
    SELECT
        shipment_id,
        MAX(exception_type) AS exception_type,
        COUNT(*) AS exception_occurrences
    FROM gold_fact_shipment_exception
    GROUP BY shipment_id
),
status_base AS (
    SELECT
        f.shipment_id,
        f.latest_status,
        COALESCE(e.exception_type, 'NO_EXCEPTION') AS exception_type,
        CASE
            WHEN UPPER(f.delivery_outcome) = 'DELIVERED' THEN 0
            ELSE DATEDIFF(CURRENT_DATE(), CAST(f.booking_ts AS DATE))
        END AS age_days
    FROM gold_fact_shipment f
    LEFT JOIN exception_rollup e
      ON f.shipment_id = e.shipment_id
)
SELECT
    CURRENT_DATE() AS snapshot_date,
    latest_status,
    CASE
        WHEN age_days <= 2 THEN '0_2_DAYS'
        WHEN age_days <= 7 THEN '3_7_DAYS'
        WHEN age_days <= 14 THEN '8_14_DAYS'
        ELSE '15_PLUS_DAYS'
    END AS ageing_band,
    exception_type,
    COUNT(DISTINCT shipment_id) AS shipment_count
FROM status_base
GROUP BY latest_status,
         CASE
            WHEN age_days <= 2 THEN '0_2_DAYS'
            WHEN age_days <= 7 THEN '3_7_DAYS'
            WHEN age_days <= 14 THEN '8_14_DAYS'
            ELSE '15_PLUS_DAYS'
         END, exception_type;

num_affected_rows,num_inserted_rows


## 7. KPI contract table

This single query provides the shipment-level KPI cards that Week 8 can reconcile back to Gold. Denominators are explicit and zero-denominator rates return `NULL` rather than a misleading zero.

In [0]:
%sql
SELECT
    COUNT(DISTINCT shipment_id) AS total_shipments,
    COUNT(DISTINCT CASE WHEN delivered_flag = 1 THEN shipment_id END) AS delivered_shipments,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN on_time_flag = 1 AND delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END), 0), 2) AS on_time_delivery_rate_pct,
    ROUND(AVG(CASE WHEN delivered_flag = 1 AND delivery_delay_minutes > 0 THEN delivery_delay_minutes END), 2) AS avg_delivery_delay_minutes_late_only,
    ROUND(AVG(CASE WHEN delivered_flag = 1 AND transit_hours IS NOT NULL THEN transit_hours END), 2) AS avg_end_to_end_transit_hours,
    COALESCE(ROUND(100.0 * (SELECT COUNT(DISTINCT shipment_id) FROM gold_fact_shipment_exception)
          / NULLIF(COUNT(DISTINCT shipment_id), 0), 2), 0) AS exception_rate_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND attempt_count = 1 THEN shipment_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND attempt_count IS NOT NULL THEN shipment_id END), 0), 2) AS first_attempt_delivery_success_rate_pct
FROM gold_fact_shipment;

total_shipments,delivered_shipments,on_time_delivery_rate_pct,avg_delivery_delay_minutes_late_only,avg_end_to_end_transit_hours,exception_rate_pct,first_attempt_delivery_success_rate_pct
99857,77929,74.76,229.62,32.22,14.81,86.09


In [0]:
%sql
SELECT
    ROUND(AVG(avg_dwell_minutes), 2) AS hub_average_dwell_minutes
FROM gold_hub_throughput_summary
WHERE avg_dwell_minutes IS NOT NULL;

hub_average_dwell_minutes
365.42


## 8. Gold grain and measure validation

These checks are the Week-7 proof layer. A zero count is the expected result for every failure query.

In [0]:
%sql
SELECT 'gold_fact_shipment_duplicate_key' AS check_name, COUNT(*) AS failures
FROM (SELECT shipment_id FROM gold_fact_shipment GROUP BY shipment_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'gold_fact_shipment_scan_duplicate_key', COUNT(*)
FROM (SELECT scan_id FROM gold_fact_shipment_scan GROUP BY scan_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'gold_fact_shipment_exception_duplicate_key', COUNT(*)
FROM (SELECT exception_id FROM gold_fact_shipment_exception GROUP BY exception_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'negative_freight', COUNT(*) FROM gold_fact_shipment WHERE freight_amount_inr < 0
UNION ALL
SELECT 'negative_weight', COUNT(*) FROM gold_fact_shipment WHERE package_weight_kg < 0
UNION ALL
SELECT 'negative_package_count', COUNT(*) FROM gold_fact_shipment WHERE package_count < 0
UNION ALL
SELECT 'negative_delivery_delay', COUNT(*) FROM gold_fact_shipment WHERE delivery_delay_minutes < 0
UNION ALL
SELECT 'invalid_on_time_flag', COUNT(*) FROM gold_fact_shipment WHERE on_time_flag NOT IN (0,1)
UNION ALL
SELECT 'invalid_first_attempt_flag', COUNT(*) FROM gold_fact_shipment WHERE first_attempt_success_flag NOT IN (0,1);

check_name,failures
gold_fact_shipment_duplicate_key,0
gold_fact_shipment_scan_duplicate_key,30
gold_fact_shipment_exception_duplicate_key,0
negative_freight,0
negative_weight,0
negative_package_count,0
negative_delivery_delay,0
invalid_on_time_flag,0
invalid_first_attempt_flag,0


In [0]:
%sql
SELECT
    'gold_shipment_delay_summary' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, carrier_id, service_level, delay_band)) AS distinct_grain_keys,
    COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, carrier_id, service_level, delay_band)) AS duplicate_grain_rows
FROM gold_shipment_delay_summary
UNION ALL
SELECT 'gold_carrier_performance_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), carrier_id, service_level)), COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), carrier_id, service_level)) FROM gold_carrier_performance_summary
UNION ALL
SELECT 'gold_route_reliability_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, service_level)), COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, service_level)) FROM gold_route_reliability_summary
UNION ALL
SELECT 'gold_hub_throughput_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(event_date AS STRING), time_band, hub_id)), COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', CAST(event_date AS STRING), time_band, hub_id)) FROM gold_hub_throughput_summary
UNION ALL
SELECT 'gold_shipment_status_exception_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(snapshot_date AS STRING), latest_status, ageing_band, exception_type)), COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', CAST(snapshot_date AS STRING), latest_status, ageing_band, exception_type)) FROM gold_shipment_status_exception_summary;

table_name,row_count,distinct_grain_keys,duplicate_grain_rows
gold_shipment_delay_summary,52381,52381,0
gold_carrier_performance_summary,4936,4936,0
gold_route_reliability_summary,17930,17930,0
gold_hub_throughput_summary,9445,9445,0
gold_shipment_status_exception_summary,47,47,0


## 9. KPI reconciliation back to the controlling fact

The daily reconciliation proves that the summary layer does not inflate or lose shipments. Expected difference is zero for shipment counts and delivered counts. On-time rate is compared using the same valid-promise denominator as the KPI contract.

In [0]:
%sql
WITH expected AS (
    SELECT
        booking_date AS metric_date,
        COUNT(DISTINCT shipment_id) AS expected_total_shipments,
        COUNT(DISTINCT CASE WHEN delivered_flag = 1 THEN shipment_id END) AS expected_delivered_shipments,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN on_time_flag = 1 AND delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END)
              / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END), 0), 2) AS expected_on_time_rate_pct
    FROM gold_fact_shipment
    GROUP BY booking_date
),
actual AS (
    SELECT
        booking_date AS metric_date,
        COUNT(DISTINCT shipment_id) AS actual_total_shipments,
        COUNT(DISTINCT CASE WHEN delivered_flag = 1 THEN shipment_id END) AS actual_delivered_shipments,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN on_time_flag = 1 AND delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END)
              / NULLIF(COUNT(DISTINCT CASE WHEN delivered_flag = 1 AND promised_delivery_ts IS NOT NULL THEN shipment_id END), 0), 2) AS actual_on_time_rate_pct
    FROM gold_fact_shipment
    GROUP BY booking_date
)
SELECT
    e.metric_date,
    e.expected_total_shipments, a.actual_total_shipments,
    e.expected_total_shipments - a.actual_total_shipments AS total_shipment_diff,
    e.expected_delivered_shipments, a.actual_delivered_shipments,
    e.expected_delivered_shipments - a.actual_delivered_shipments AS delivered_diff,
    e.expected_on_time_rate_pct, a.actual_on_time_rate_pct,
    ROUND(e.expected_on_time_rate_pct - a.actual_on_time_rate_pct, 2) AS on_time_rate_diff
FROM expected e
JOIN actual a ON e.metric_date = a.metric_date
ORDER BY e.metric_date;

metric_date,expected_total_shipments,actual_total_shipments,total_shipment_diff,expected_delivered_shipments,actual_delivered_shipments,delivered_diff,expected_on_time_rate_pct,actual_on_time_rate_pct,on_time_rate_diff
2026-01-01,580,580,0,445,445,0,75.28,75.28,0.00
2026-01-02,570,570,0,435,435,0,76.55,76.55,0.00
2026-01-03,485,485,0,372,372,0,74.73,74.73,0.00
2026-01-04,566,566,0,440,440,0,72.73,72.73,0.00
2026-01-05,560,560,0,431,431,0,77.03,77.03,0.00
2026-01-06,528,528,0,410,410,0,69.02,69.02,0.00
2026-01-07,518,518,0,426,426,0,74.18,74.18,0.00
2026-01-08,589,589,0,452,452,0,71.02,71.02,0.00
2026-01-09,526,526,0,407,407,0,76.90,76.90,0.00
2026-01-10,538,538,0,420,420,0,73.10,73.10,0.00


## 10. Manual spot checks

The playbook requires actual records, including an edge case. The queries below expose shipment, scan/dwell and exception examples for screenshot/evidence capture.

In [0]:
%sql
-- Shipment-level spot check: one delivered/on-time example when available.
SELECT *
FROM gold_fact_shipment
WHERE delivered_flag = 1
ORDER BY shipment_id
LIMIT 1;

source_record_id,shipment_id,order_reference,route_id,origin_hub_id,destination_hub_id,carrier_id,service_level,transport_mode,booking_ts,pickup_ts,promised_delivery_ts,actual_delivery_ts,return_completed_ts,latest_status,delivery_outcome,status_reason,package_weight_kg,package_count,freight_amount_inr,attempt_count,exception_flag,delivery_delay_minutes,transit_hours,on_time_flag,first_attempt_success_flag,delivered_flag,booking_date,delivery_date
SHPREC00000001,SHP00000001,ORD000000001,R065,H013,H011,C006,STANDARD,ROAD,2026-05-06T09:30:00.000Z,2026-05-06T18:59:30.855Z,2026-05-09T00:35:30.855Z,2026-05-08T13:15:50.055Z,null,DELIVERED,DELIVERED,,8.16,3,1072.54,1,false,0,42.272222,1,1,1,2026-05-06,2026-05-08


In [0]:
%sql
-- Edge case: delivered shipment with a late delivery, if present.
SELECT shipment_id, promised_delivery_ts, actual_delivery_ts, delivery_delay_minutes, attempt_count, exception_flag
FROM gold_fact_shipment
WHERE delivered_flag = 1 AND delivery_delay_minutes > 0
ORDER BY delivery_delay_minutes DESC
LIMIT 1;

shipment_id,promised_delivery_ts,actual_delivery_ts,delivery_delay_minutes,attempt_count,exception_flag
SHP00045770,2026-03-21T04:28:11.600Z,2026-03-22T21:58:46.280Z,2490,1,false


In [0]:
%sql
-- Hub dwell example.
SELECT *
FROM gold_hub_throughput_summary
WHERE avg_dwell_minutes IS NOT NULL
ORDER BY avg_dwell_minutes DESC
LIMIT 1;

event_date,time_band,hub_id,shipment_count,avg_dwell_minutes
2026-07-01,18_24,H005,1,1169.60


In [0]:
%sql
-- Exception example.
SELECT *
FROM gold_fact_shipment_exception
ORDER BY exception_ts
LIMIT 1;

exception_id,shipment_id,exception_type,exception_ts,hub_id,severity,resolution_status,resolution_time,scan_id,exception_date
EXC00006936,SHP00036980,OTHER,2026-01-01T10:32:19.000Z,H006,MEDIUM,RESOLVED,2026-01-01T17:17:31.000Z,SCN000263071,2026-01-01


## 11. Controlled rerun proof

All Gold writes use `CREATE OR REPLACE TABLE ... USING DELTA`, so rerunning the notebook from the same Trusted inputs should preserve the same business grains and counts. Capture the result of this cell **before and after a second full notebook run** and confirm the differences remain zero.

In [0]:
%sql
SELECT 'gold_fact_shipment' AS table_name, COUNT(*) AS row_count, COUNT(DISTINCT shipment_id) AS distinct_business_keys FROM gold_fact_shipment
UNION ALL SELECT 'gold_fact_shipment_scan', COUNT(*), COUNT(DISTINCT scan_id) FROM gold_fact_shipment_scan
UNION ALL SELECT 'gold_fact_shipment_exception', COUNT(*), COUNT(DISTINCT exception_id) FROM gold_fact_shipment_exception
UNION ALL SELECT 'gold_shipment_delay_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, carrier_id, service_level, delay_band)) FROM gold_shipment_delay_summary
UNION ALL SELECT 'gold_hub_throughput_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(event_date AS STRING), time_band, hub_id)) FROM gold_hub_throughput_summary
UNION ALL SELECT 'gold_carrier_performance_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), carrier_id, service_level)) FROM gold_carrier_performance_summary
UNION ALL SELECT 'gold_route_reliability_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(delivery_date AS STRING), route_id, service_level)) FROM gold_route_reliability_summary
UNION ALL SELECT 'gold_shipment_status_exception_summary', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', CAST(snapshot_date AS STRING), latest_status, ageing_band, exception_type)) FROM gold_shipment_status_exception_summary;

table_name,row_count,distinct_business_keys
gold_fact_shipment,99857,99857
gold_fact_shipment_scan,710666,710636
gold_fact_shipment_exception,18330,18330
gold_shipment_delay_summary,52381,52381
gold_hub_throughput_summary,9445,9445
gold_carrier_performance_summary,4936,4936
gold_route_reliability_summary,17930,17930
gold_shipment_status_exception_summary,47,47


## 12. Week 7 exit evidence

### Week 7 exit evidence
- [ ] Trusted source resolution recorded
- [ ] `gold_fact_shipment` is one row per shipment
- [ ] `gold_fact_shipment_scan` is one row per scan
- [ ] `gold_fact_shipment_exception` is one row per exception
- [ ] All five approved summaries exist
- [ ] KPI contracts are explicit and denominator-safe
- [ ] Summary grain uniqueness checks return zero failures
- [ ] KPI reconciliation has zero count differences
- [ ] Three or more actual records, including an edge case, are captured
- [ ] Full notebook rerun produces identical row/key counts
- [ ] Week 7 log and AI transparency note are updated

**Do not claim Week 7 complete until the executed checks above are captured as evidence.**

In [0]:
# Optional Python evidence summary for the Databricks run log.
gold_tables = [
    "gold_fact_shipment",
    "gold_fact_shipment_scan",
    "gold_fact_shipment_exception",
    "gold_shipment_delay_summary",
    "gold_hub_throughput_summary",
    "gold_carrier_performance_summary",
    "gold_route_reliability_summary",
    "gold_shipment_status_exception_summary"
]
for table in gold_tables:
    df = spark.table(table)
    print(f"{table}: rows={df.count()}")

gold_fact_shipment: rows=99857
gold_fact_shipment_scan: rows=710666
gold_fact_shipment_exception: rows=18330
gold_shipment_delay_summary: rows=52381
gold_hub_throughput_summary: rows=9445
gold_carrier_performance_summary: rows=4936
gold_route_reliability_summary: rows=17930
gold_shipment_status_exception_summary: rows=47
